In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [3]:
load_dotenv()
model = ChatGoogleGenerativeAI(model='gemini-2.0-flash-lite')

In [19]:
class SentimentSchema(BaseModel):

    sentiment: Literal['positive', 'negative'] = Field(description='Sentiment of the review')

class DiagnosisSchema(BaseModel):

        issue_type: Literal['UX', 'Performance', 'Bug', 'Support', 'Other'] = Field(description='The category of issue mentioned in the review')
        tone: Literal['angry', 'frustrated', 'disappointed', 'calm'] = Field(description='The emotional tone expressed by the user')
        urgency: Literal['low', 'medium', 'high'] = Field(description='How urgent or critical the issue appears to be') 

In [20]:
s_m = model.with_structured_output(SentimentSchema)
s_m2 = model.with_structured_output(DiagnosisSchema)

In [6]:
prompt = 'What is the sentiment of the following review - THe software is good'
s_m.invoke(prompt).sentiment

'positive'

In [7]:
class ReviewState(TypedDict):

    review: str
    sentiment: Literal['positive', 'negative']
    diagnose: dict
    response: str

In [21]:
def find_sentiment(state: ReviewState):

    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    sentiment = s_m.invoke(prompt).sentiment

    return {'sentiment': sentiment}

def check_sentiment(state: ReviewState) -> Literal['positive_response', 'run_diagnosis']:

    if state['sentiment'] == 'positive':
        return 'positive_response'
    else: 
        return 'run_diagnose'

def positive_response(state: ReviewState):

    prompt = f"""
    Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
    Also, kindly ask the user to leave feedback on our website.
    """            

    response = model.invoke(prompt).content

    return {'response': response}

def run_diagnosis(state: ReviewState):

    prompt = f"""
    Diagnose this negative review: \n\n{state['review']}\n"
    "Return isuue_type, tone and urgency.
    """ 

    response = s_m2.invoke(prompt)

    return {'diagnosis': response.model_dump()}  

def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""
    You are a support assistant.
    The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
    Write an emphateic, helpful resolution message. 
    """

    response = model.invoke(prompt).content

    return {'response': response}     


In [25]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()
# graph.compile()

In [26]:
initial_stage = {
    'review': 'Absolutely loved the experience! Everything went smoothly from start to finish, and the quality exceeded my expectations. Would definitely recommend it to anyone looking for something reliable and well-done.'
}

workflow.invoke(initial_stage)

{'review': 'Absolutely loved the experience! Everything went smoothly from start to finish, and the quality exceeded my expectations. Would definitely recommend it to anyone looking for something reliable and well-done.',
 'sentiment': 'positive',
 'response': 'Thank you so much for taking the time to share this wonderful review. We’re thrilled to hear that your experience was smooth from start to finish and that the quality exceeded your expectations. Your willingness to recommend us means the world to us.\n\nIf you have a moment, we’d be grateful if you could also leave your feedback on our website in the Reviews section. Sharing your thoughts helps others feel confident about choosing us.\n\nWith appreciation,\nThe [Your Company] Team'}